# 01 AI Agent 工程全景

本章先给出全局坐标，再进入后续各章。

**结论：AI Agent 是一个持续运行、调用外部能力并接受验证的工程系统。** LLM 是决策引擎；Tokenization 决定模型接收的信息表示；RAG、Memory、Skills、Spec 和 Wiki 提供上下文；Agent Loop 推进任务；Harness 负责把这些模块装配成可恢复、可观测、受权限约束的运行时；MCP、CLI、API 等接口把决策连接到真实工具。

完成本章后，你应该能够：

1. 概括通用大模型出现前 AI 已经具备的主要能力，以及这些能力尚未统一的原因。
2. 沿一条主链解释 Agent 从用户目标到真实动作的全过程。
3. 区分模型、框架、协议、平台、产品和算力，避免把品牌名单当成系统架构。
4. 说明 AutoGen、LangGraph、CrewAI、RAG、Tokenization、GigaTok 等概念各自处于哪一层。
5. 用最小 Planner 观察 route、risk、approval 和 next action。

## 1. 大模型之前：AI 已经发展到什么程度

**结论：通用大模型继承并统一了此前分散在规则推理、搜索、感知、预测和语言处理系统中的能力。** 早期 AI 已经能在边界明确的任务中做出高质量判断，但不同能力通常由不同模型、知识库和工程系统分别实现。

本节把 **2018—2022 年**视为过渡带：BERT 和 GPT-3 已经体现大规模预训练与跨任务迁移，InstructGPT 又把人类反馈引入指令对齐。[44][45][46] 如果按预训练模型划线，通用大模型时代大约从 2018 年开始；如果按 ChatGPT 类自然语言产品划线，转折点更接近 2022 年末。[47]

| 阶段 | 代表进展 | 当时已经能做到什么 | 主要局限 |
| --- | --- | --- | --- |
| 1940s—1950s：理论奠基 | 图灵提出模仿游戏；达特茅斯研究计划正式使用“人工智能”一词，并把语言、抽象、问题求解和自我改进列为研究目标。[36][37] | 把“机器智能”转化为可研究的问题，建立计算、逻辑和神经元模型等基本路线 | 主要停留在理论、小型程序和受限演示 |
| 1950s—1970s：符号主义与搜索 | 用符号、规则、逻辑和启发式搜索表达问题；Logic Theory Machine 已能寻找初等符号逻辑的定理证明。[48] | 在定理证明、博弈和规划等形式化环境中完成明确任务 | 知识主要靠人工编码，问题规模增大后搜索空间迅速膨胀 |
| 1970s—1980s：专家系统 | MYCIN 把专家规则、咨询、解释和规则获取组合成医疗决策支持系统。[38] | 在狭窄专业领域复现部分专家判断，并解释使用了哪些规则 | 知识获取和维护成本高，领域变化后需要人工更新规则 |
| 1980s—2000s：统计学习与神经网络 | 神经网络重新发展，统计模型进入语音、文字识别和分类系统；LeNet 展示了端到端文档识别，Deep Blue 在 1997 年击败国际象棋世界冠军。[39][40] | 从数据中学习模式，并在边界明确的识别、预测和搜索任务中达到实用水平 | 每类任务仍需要专门的数据、特征、目标函数和模型 |
| 2012—2016：深度学习突破 | AlexNet 依靠大规模数据、GPU 和深层卷积网络显著降低 ImageNet 识别错误；AlphaGo 组合深度神经网络与树搜索掌握围棋。[41][42] | 自动学习图像、语音和复杂状态的表示，在部分基准与封闭博弈中超过传统方法或顶尖人类 | 依赖大量数据与算力，模型通常只解决一种输入形式或任务 |
| 2017：Transformer | Transformer 用 Attention 替代循环结构，在机器翻译中提高质量和并行训练效率。[43] | 更有效地建模长序列，为扩大模型、数据和训练规模提供统一架构 | 当时仍以翻译等特定训练任务为主 |
| 2018—2020：预训练模型 | BERT 通过预训练加微调覆盖多种理解任务；GPT-3 展示无需梯度更新的少样本任务适配。[44][45] | 一个预训练模型开始复用于问答、分类、生成、翻译等多种语言任务 | 任务表现受提示方式影响，事实可靠性和指令遵循仍不稳定 |
| 2020—2022：通用模型前夜 | 模型规模、上下文学习和人类反馈训练汇合；InstructGPT 改善了模型遵循用户意图的能力。[46] | 用户开始通过自然语言调用同一个模型完成写作、问答、总结和部分代码任务 | 仍会编造事实、误解目标，也缺少可靠的工具、状态、权限和验证闭环 |

在通用大模型出现前，AI 已经形成五类成熟能力：

1. **规则推理与搜索**：在规则明确的空间中证明、规划或选择动作。[37][48]
2. **感知**：识别文字、语音、图像和结构化信号。[39][41]
3. **预测与分类**：从样本中学习输入与标签之间的统计关系。[39][41]
4. **专业决策支持**：在医疗等窄领域复用专家知识并解释规则。[38]
5. **语言处理**：分别完成翻译、分类、问答和生成。[43][44][45]

这些能力的共同限制是复用单位仍然偏小：换一个任务，通常就要更换模型、数据、规则或系统。大规模预训练把复用单位提升为“一个基础模型处理多种任务”，自然语言成为统一入口；Agent 工程继续补上外部知识、工具执行、状态恢复、权限控制和结果验证。下面的全景图从这条分界线继续展开。

## 2. 一张图看懂 AI Agent 工程

下图是 **6400 × 4480** 总览版。Notebook 会按页面宽度缩放；需要查看节点文字时，可以点击图片或下面的原图链接放大。

[![AI Agent 工程全景图](../assets/agent-engineering-panorama.png)](../assets/agent-engineering-panorama.png)

[打开 6400 × 4480 PNG 原图](../assets/agent-engineering-panorama.png) · [打开可无限缩放的 SVG](../assets/agent-engineering-panorama.svg)

### 2.1 主运行链放大图

[![AI Agent 主运行链放大图](../assets/agent-engineering-main-chain.png)](../assets/agent-engineering-main-chain.png)

### 2.2 框架与供给生态放大图

[![Agent 框架与供给生态放大图](../assets/agent-framework-ecosystem.png)](../assets/agent-framework-ecosystem.png)

图中绿色实线是一次任务的主路径；紫色虚线表示框架、模型与状态对运行时的支撑；红色边界是 Agent Harness；黄色边界是工程实现与供给生态。生态名称用于定位，不表示它们互为同类，也不表示推荐顺序。图中产品与项目状态核对日期为 **2026-07-27**。

### 2.3 主运行链说明

1. **用户目标 / 事件**从 Chat、API、IDE、App 或定时任务进入系统。
2. **Harness**装配 Prompt、Spec、Wiki、Skills、RAG 结果、Memory、权限和预算。
3. 文本、图像或音频先经过 **Tokenization**，再进入 **LLM**。模型理解任务、生成候选动作或发出 Tool Call。
4. **Agent Loop**执行 `Plan → Act → Observe → Verify → Replan / Stop / Handoff`。一次模型回复只是 Loop 中的一步。
5. **Tool Calling、MCP、CLI、API、Browser、A2A**提供能力接口；Shell、文件、数据库、SaaS 和浏览器承担真实执行。
6. **State / Checkpoint**保存可恢复状态；**Evals、Tracing、Guardrails、沙箱、权限、人审和审计**决定系统能否安全上线。
7. 系统交付可验证的回答、文件、PR、通知或业务状态变更，并把结果反馈给 Loop。

一个完整的 Claude Code、Codex 或 OpenCode 产品通常覆盖入口、上下文装配、Loop、工具、状态与安全边界；它们不是单独的模型或单独的 CLI。

## 3. 七层分类法

| 层级 | 回答的问题 | 关键对象 |
| --- | --- | --- |
| 1. 表示与模型 | 信息怎样进入模型，模型怎样推理？ | Tokenization、Tokenizer、LLM、视觉 Tokenizer |
| 2. 上下文与知识 | 本次推理需要哪些事实、规则和能力说明？ | Prompt、Spec、Wiki、RAG、Memory、Skills、项目规则 |
| 3. 决策循环 | 下一步做什么，何时重试、停止或转人工？ | Agent Loop、Planner、State Machine、Handoff |
| 4. 运行时外壳 | 如何装配上下文、执行工具、恢复状态和控制权限？ | Agent Harness、Checkpoint、Budget、Sandbox、Approval |
| 5. 能力接口 | 模型如何连接外部世界？ | Tool Calling、MCP、CLI、API、Browser、Computer Use、A2A |
| 6. 工程框架与产品 | 谁帮助构建或直接提供完整 Agent？ | LangGraph、CrewAI、AutoGen、Claude Code、Codex、OpenCode |
| 7. 供给与生产 | 模型、托管、算力、评测和监控从哪里来？ | Hugging Face、百炼、方舟、AutoDL、Evals、Tracing、Guardrails |

分类的价值在于判断依赖关系：框架实现 Loop 和状态语义，模型提供推理，平台提供托管服务，算力承载运行，产品则把多层能力组合成可直接使用的体验。

### 3.1 Tokenization、LLM 与 RAG

**Tokenization**把原始输入编码成模型能够处理的 token 序列。Token 数量影响上下文占用、延迟与费用；词表、切分规则和多模态编码方式会影响模型看到的信息边界。GigaTok 属于视觉 Tokenizer 研究与实现方向，位置靠近多模态表示层，不属于 Agent 框架。[1]

**LLM**根据当前 token 上下文预测输出，可以生成自然语言、结构化数据或工具调用参数。它不天然拥有企业最新数据、业务权限和可靠的长期任务状态。

**RAG**在推理前检索外部资料，把相关片段及来源放入上下文。[2] 一个可上线的 RAG 还需要数据清洗、chunk、embedding、metadata 权限过滤、rerank、引用和召回评测。RAG 是上下文供给链，不等于 Agent；Agent 可以调用 RAG，也可以在无需检索时直接回答或调用工具。

### 3.2 Skill、MCP、CLI、Loop 与 Harness

| 概念 | 最小定义 | 不负责什么 |
| --- | --- | --- |
| Skill | 可发现、可复用的能力包，通常包含说明、脚本、参考资料和资产；Agent Skills 以 `SKILL.md` 为核心规范。[3] | 不定义通用网络传输协议 |
| MCP | Host、Client、Server 之间交换 tools、resources、prompts 的上下文协议。[4] | 不替 Agent 决定任务目标和停止条件 |
| CLI | 人或 Agent 调用本地程序的命令行入口 | 不自动提供语义发现、权限治理和状态恢复 |
| Agent Loop | 推进 Plan、Act、Observe、Verify、Replan、Stop、Handoff 的控制循环 | 不天然包含部署、追踪和沙箱 |
| Harness | 围绕模型与 Loop 的运行时外壳：上下文装配、工具执行、状态恢复、权限、预算、反馈 | 不等于某一个框架或某一个模型 |
| Spec / Wiki | 任务契约与共享知识：目标、约束、接口、决策和验收标准 | 不能替代可执行验证与权限控制 |

**Agentic Wiki**可以理解为面向 Agent 和人的持续知识协作形态；**Spec-driven** 强调先把行为契约与验收标准写清，再让 Agent 执行。它们都进入上下文层，但必须接受代码、测试、数据权限和真实环境的验证。

## 4. 生态全景与产品定位

| 类别 | 代表项目 / 产品 | 在系统中的位置 |
| --- | --- | --- |
| 通用模型 | OpenAI GPT、Anthropic Claude、Google Gemini、DeepSeek、Qwen、GLM、豆包、Kimi、MiniMax | 推理与生成 |
| 模型与开发平台 | OpenAI Platform、Anthropic Claude Platform、Google AI / Vertex AI、阿里云百炼、火山方舟、智谱开放平台 | API、模型托管、工具与应用服务 |
| 模型生态与推理 | Hugging Face Hub、ModelScope、Ollama、vLLM、OpenRouter | 模型、数据集、Demo、下载与推理 |
| 算力 | AutoDL、云 GPU、本地 GPU | 训练、微调、推理与实验环境 |
| Agent 框架 / SDK | PydanticAI、LangGraph、LlamaIndex、Haystack、OpenAI Agents SDK、Claude Agent SDK、Google ADK、Strands、Microsoft Agent Framework、smolagents、Agno | Loop、状态、工具、工作流与运行时抽象 |
| 多 Agent 框架 | CrewAI、AutoGen、AG2、CAMEL-AI、Langroid、BeeAI、MetaGPT | 角色、协作、消息、委派和多运行时组合 |
| TypeScript Agent 生态 | Mastra、Vercel AI SDK、LangChain.js，以及 Google ADK、Strands、BeeAI 的 TypeScript 实现 | Web / Node.js 应用、流式 UI、Agent 与 Workflow |
| 状态与记忆专项 | Letta | 持久化 Agent、可编辑记忆与共享状态 |
| 低代码应用平台 | Dify、Coze Studio、Langflow、Flowise、n8n、Microsoft Copilot Studio | 可视化编排、连接器、发布与运营 |
| Coding Agent 产品 | Claude Code、Codex、OpenCode、Cursor、GitHub Copilot、Cline、Aider、OpenHands | 面向代码库的完整 Agent 体验 |
| 协议与规范 | MCP、Agent Skills、A2A、AG-UI、JSON Schema、OpenAPI | 能力发现、互操作、界面事件与数据契约 |
| 多模态 | 可灵、图像 / 视频模型、GigaTok | 内容生成与视觉 token 表示 |

这张表按“它在系统中承担什么责任”分类。一个项目可能跨层，例如 LlamaIndex 同时覆盖 RAG、Agent 和 Workflow；Google ADK、Strands、BeeAI 同时支持多语言；Letta 既有 SDK，也提供运行平台。分类采用主能力，不强行把项目限制在单一标签。

Hugging Face Hub 提供模型、数据集和 Spaces 等协作资产。[5] 百炼、方舟和智谱开放平台属于模型与应用服务层；AutoDL 属于算力供给层。[6][7][8][9] 可灵是视频生成产品，GigaTok 是视觉 Tokenizer，二者都与多模态相关，但承担不同职责。[1][10]

### 4.1 Agent 框架全景

框架数量很多，先按**控制权、状态位置和主要语言**分类，再选择具体项目。下表收录截至 2026-07-27 仍有明确官方资料、并对当前工程实践有代表性的框架与 SDK。

| 家族 | 代表项目 | 主要抽象与适用方向 |
| --- | --- | --- |
| 显式图与可恢复工作流 | LangGraph、Microsoft Agent Framework Workflows、LlamaIndex Workflows | State、Node / Executor、Edge、Event、Interrupt、Checkpoint；适合长流程和恢复语义 |
| 类型化 / 轻量 Agent SDK | PydanticAI、OpenAI Agents SDK、Claude Agent SDK、smolagents、Agno | Agent、Tool、结构化输出、依赖、Handoff / Team；适合用较少抽象组装 Loop |
| 数据、RAG 与 Pipeline | LlamaIndex、Haystack、LangChain | 检索、索引、Pipeline、Tool 与 Agent；适合知识密集型应用 |
| 多 Agent 协作 | CrewAI、AutoGen、AG2、CAMEL-AI、Langroid、BeeAI、MetaGPT | Role、Crew / Society、Conversation、Delegation；适合研究协作模式和角色分工 |
| 厂商通用 ADK | Google ADK、Strands Agents、Microsoft Agent Framework、OpenAI Agents SDK、Claude Agent SDK | 模型、工具、工作流、评测和部署集成；通常与厂商平台结合较深 |
| TypeScript / Web 原生 | Mastra、Vercel AI SDK、LangChain.js，以及 Google ADK、Strands、BeeAI 的 TypeScript 包 | Node.js、流式响应、前端状态、Workflow 和 Web 部署 |
| 状态与长期记忆专项 | Letta | Stateful Agent、Memory Block、共享记忆、Agent 状态序列化 |

官方资料反映了这些差异：

- LangGraph强调长运行、有状态、可恢复的图编排；PydanticAI强调类型契约；OpenAI Agents SDK提供 Runner、Tools、Guardrails、Handoffs 和 Sessions。[11][18][22]
- Claude Agent SDK把 Claude Code 的 Agent Loop、工具执行和上下文管理能力开放给 Python / TypeScript 应用；smolagents提供较小的 Agent 抽象和 CodeAgent；Agno同时覆盖 Agent、Team、Workflow 与运行服务。[19][23][26]
- LlamaIndex和Haystack从数据、RAG、Pipeline延伸到 Tool-using Agent；它们与纯多 Agent 框架的起点不同。[24][25]
- Google ADK支持 Python、TypeScript、Go、Java、Kotlin，并提供图工作流、多 Agent、评测与部署；Strands支持 Python 和 TypeScript；Microsoft Agent Framework统一 Agent 与 Workflow 抽象。[20][21][35]
- CrewAI以 Crews / Flows 组织协作；CAMEL-AI提供 Agent、Society、Interpreter、Memory 和 RAG；BeeAI同时提供 Python / TypeScript 多 Agent 框架。[12][28][27]
- Mastra是 TypeScript Agent 框架；Vercel AI SDK覆盖多模型、流式 UI、Agent 和长运行 Workflow；Letta把持久记忆与 Agent 状态作为核心抽象。[29][30][31]
- AG2是独立发展的多 Agent 框架；Langroid与MetaGPT分别代表消息驱动的多 Agent 编程和软件团队角色化协作。[32][33][34]

**版本与继承关系：** Microsoft 的 AutoGen 仓库当前处于维护阶段，官方建议新项目评估 Microsoft Agent Framework；Semantic Kernel 与 AutoGen 的既有概念仍值得学习，但新选型要同时检查迁移路线。[13][14][35] AG2与 Microsoft AutoGen 已是不同项目，不能当作同一个包的两个名称。[32]

**邻接工具不混入框架表：** Dify、Coze Studio、Langflow、Flowise、n8n 和 Copilot Studio 属于应用平台；MCP、A2A、Agent Skills 和 AG-UI 属于协议或规范；Claude Code、Codex、OpenCode、OpenHands 属于完整产品。DSPy更接近语言模型程序的声明与优化工具。这些项目可以与框架组合，但选型维度不同。

第 11 章只用 PydanticAI、LangGraph 和 Microsoft Agent Framework 做同题实验，因为三者分别突出类型契约、显式状态图和 Agent / Workflow 统一抽象。**教学选取三个框架，不代表生态只包含三个框架。**

### 4.2 Claude Code、Codex 与 OpenCode

这些 Coding Agent 的共同骨架是：

1. 读取仓库规则、代码、终端输出和任务上下文。
2. 规划修改并通过 CLI、文件系统、测试、浏览器或 MCP 执行动作。
3. 在权限和沙箱边界内运行，记录结果，失败时重规划。
4. 把 diff、测试、提交或 PR 作为可验证产物。

差异主要落在模型选择、交互体验、权限策略、扩展机制、会话状态和托管方式。Claude Code、Codex 和 OpenCode 的官方资料分别说明了各自的运行方式与扩展入口。[15][16][17] 因此，“Coding Agent”是完整产品类别；“CLI”只是其中一个入口。

## 5. 常见系统类型

| 类型 | 控制方式 | 典型用途 | 主要风险 |
| --- | --- | --- | --- |
| Chatbot | 单轮或多轮生成 | FAQ、解释、内容草稿 | 幻觉、上下文漂移 |
| RAG | 检索后生成 | 企业知识库、政策查询 | 召回遗漏、权限泄漏、过期数据 |
| Workflow | 预定义步骤与分支 | 审批、数据同步、固定工单 | 流程僵化、异常分支缺失 |
| Agent | 模型在约束内动态选动作 | 研发助手、售后处理、复杂资料整理 | 不确定路径、工具副作用、失控循环 |
| Multi-Agent | 多个角色或运行时协作 | 复杂拆分、并行研究、跨域交接 | 通信成本、角色重叠、错误放大 |

真实项目通常是组合系统：Workflow 固定安全边界，Agent 处理开放决策，RAG 提供证据，人工审批保护高风险动作。

## 6. AI Agent 工程师的能力结构

1. **后端工程**：Python / TypeScript、HTTP、数据库、异步任务、测试、日志、容器与部署。
2. **模型基础**：Tokenization、上下文窗口、采样、结构化输出、工具调用和多模态。
3. **上下文工程**：Prompt、Spec、Wiki、RAG、Memory、Skills 与上下文压缩。
4. **工具工程**：Schema、幂等、超时、重试、错误分类、权限和副作用控制。
5. **Loop 与状态**：计划、执行、观察、验证、停止、交接、checkpoint 和恢复。
6. **Harness**：运行隔离、预算、并发、队列、会话、artifact、receipt 和审批。
7. **评测与可观测性**：eval dataset、任务成功率、tool accuracy、trace、成本和延迟。
8. **安全与治理**：prompt injection、防越权、敏感数据、沙箱、人审和审计。
9. **产品与协作**：任务边界、失败体验、运营指标、Spec、文档和跨团队交付。

多 Agent 属于后续能力。单 Agent 的工具契约、状态与评测尚未稳定时，增加角色通常只会扩大调试面。

## 7. 课程导航

本表严格投影根目录 `README.md` 的 14 章目录；章节名称、编号和顺序保持一致。详细学习结果来自 `docs/roadmap.md`。

| 章节 | 课程主题 | 本章在全景图中的位置 |
| --- | --- | --- |
| 01 | AI Agent 工程全景 | 建立系统主链、生态分层和课程坐标 |
| 02 | LLM 初识 | Tokenization、模型训练、推理与生成 |
| 03 | 面向产研协作助手 V0 的提示词工程 | Prompt、指令边界、输出 Schema 与回归评测 |
| 04 | Function Calling | Tool Schema、模型选工具、应用执行与结果回传 |
| 05 | MCP | Host、Client、Server、Tools、Resources、Prompts |
| 06 | RAG 知识库工程 | Ingestion、Chunk、Embedding、检索、Rerank、引用与 ACL |
| 07 | 上下文工程 | Context、Memory、State、压缩与恢复 |
| 08 | Agent Loop | Plan、Act、Observe、状态机、重试与停止条件 |
| 09 | Agent Skills 与 Agent-facing CLI | 可复用工作流、渐进披露、结构化执行接口与授权 |
| 10 | Agent Harness Engineering | 上下文装配、工具运行、权限、沙箱、状态、预算与反馈 |
| 11 | Agent Frameworks | 框架内核、统一 PoC、故障注入与生产选型 |
| 12 | 生产控制 | 评测、追踪、注入防护、审批与发布门禁 |
| 13 | Capstone | 组合前 12 章能力交付完整项目 |
| 14 | 行业 Agent 落地 | 行业选型、数据闭环、风险分级和上线阶梯 |

核心依赖在第 13 章闭合；第 14 章是独立扩展，不作为前 13 章的必修依赖。

## 8. Minimal Planner

下面用不调用 LLM 的最小代码观察 Agent 控制面。它只判断：

1. 是否需要知识检索。
2. 是否需要工具调用。
3. 是否必须人工审批。
4. 风险等级和下一步动作。

这个例子刻意保持简单，用于固定 route 和 safety contract；后续章节再逐步替换为 RAG、Tool Calling、状态机和真实模型。

### 8.1 定义计划结构

计划至少要让运行时知道任务、路由、依赖能力、风险和下一步动作。

In [1]:
from dataclasses import asdict, dataclass
import json


@dataclass(frozen=True)
class Plan:
    # frozen=True 防止计划在传递过程中被意外修改。
    task: str
    route: str
    needs: list[str]
    risk_level: str
    next_action: str


### 8.2 配置任务识别规则

规则版 Planner 便于观察控制流程。它的判断能力很弱，但预期行为可以被测试固定。

In [2]:
KNOWLEDGE_KEYWORDS = (
    "总结", "查询", "资料", "材料", "知识库", "文档", "纪要", "README", "病例", "合同"
)

TOOL_KEYWORDS = (
    "读取", "计算", "创建", "发送", "写入", "更新", "删除", "开平方", "上传", "下载"
)

HUMAN_REVIEW_KEYWORDS = (
    "删除", "发送", "付款", "审批", "发布", "诊断", "医疗", "合同", "转账"
)


def contains_any(text: str, keywords: tuple[str, ...]) -> bool:
    # 统一转成小写，兼容 README 这类英文关键词。
    normalized = text.lower()
    return any(keyword.lower() in normalized for keyword in keywords)


### 8.3 实现 Planner

输入是用户任务；输出是任务路线、所需能力、风险等级和下一步动作。高风险动作先产出草案并等待批准。

In [3]:
def make_plan(task: str) -> Plan:
    needs: list[str] = []

    # 每条规则只声明一种能力需求，组合路由由后面统一决定。
    if contains_any(task, KNOWLEDGE_KEYWORDS):
        needs.append("knowledge_retrieval")

    if contains_any(task, TOOL_KEYWORDS):
        needs.append("tool_calling")

    if contains_any(task, HUMAN_REVIEW_KEYWORDS):
        needs.append("human_review")

    if not needs:
        needs.append("direct_answer")

    # 风险优先级高于执行便利性：需要人审时不得直接执行工具。
    if "human_review" in needs:
        risk_level = "high"
        next_action = "Prepare a draft plan and wait for human approval before taking action."
    elif "tool_calling" in needs:
        risk_level = "medium"
        next_action = "Validate tool parameters, call the required tool, then summarize the result."
    elif "knowledge_retrieval" in needs:
        risk_level = "low"
        next_action = "Retrieve relevant context, cite sources, then answer."
    else:
        risk_level = "low"
        next_action = "Answer directly."

    route = "agentic_workflow" if len(needs) > 1 else needs[0]

    return Plan(
        task=task,
        route=route,
        needs=needs,
        risk_level=risk_level,
        next_action=next_action,
    )


### 8.4 运行样例

观察同一个 Planner 如何为直接回答、RAG、工具调用和高风险组合任务选择不同路线。

In [4]:
tasks = [
    "解释一下 AI Agent 是什么",
    "总结 README.md 的主要内容",
    "计算 16 的开平方",
    "读取会议纪要并发送团队通知",
]

for task in tasks:
    plan = make_plan(task)
    print(json.dumps(asdict(plan), ensure_ascii=False, indent=2))
    print("-" * 60)


{
  "task": "解释一下 AI Agent 是什么",
  "route": "direct_answer",
  "needs": [
    "direct_answer"
  ],
  "risk_level": "low",
  "next_action": "Answer directly."
}
------------------------------------------------------------
{
  "task": "总结 README.md 的主要内容",
  "route": "knowledge_retrieval",
  "needs": [
    "knowledge_retrieval"
  ],
  "risk_level": "low",
  "next_action": "Retrieve relevant context, cite sources, then answer."
}
------------------------------------------------------------
{
  "task": "计算 16 的开平方",
  "route": "tool_calling",
  "needs": [
    "tool_calling"
  ],
  "risk_level": "medium",
  "next_action": "Validate tool parameters, call the required tool, then summarize the result."
}
------------------------------------------------------------
{
  "task": "读取会议纪要并发送团队通知",
  "route": "agentic_workflow",
  "needs": [
    "knowledge_retrieval",
    "tool_calling",
    "human_review"
  ],
  "risk_level": "high",
  "next_action": "Prepare a draft plan and wait for human approva

### 8.5 验证关键行为

断言把路由与风险规则变成可回归的契约。后续即使把关键词替换成 LLM 分类器，这些安全预期也不应悄悄改变。

In [5]:
direct_plan = make_plan("解释一下 AI Agent 是什么")
assert direct_plan.needs == ["direct_answer"]
assert direct_plan.risk_level == "low"

tool_plan = make_plan("计算 16 的开平方")
assert "tool_calling" in tool_plan.needs
assert tool_plan.risk_level == "medium"

review_plan = make_plan("读取会议纪要并发送团队通知")
assert "knowledge_retrieval" in review_plan.needs
assert "tool_calling" in review_plan.needs
assert "human_review" in review_plan.needs
assert review_plan.risk_level == "high"
assert "approval" in review_plan.next_action

print("All checks passed.")


All checks passed.


### 8.6 当前版本的失败模式

1. 关键词规则无法稳定识别同义词、否定句、隐含意图和复杂任务拆解。
2. “发送一份不要真的发送的草稿”仍可能触发人审；规则不了解语义范围。
3. Planner 没有真正执行 RAG、工具、checkpoint 或权限检查。
4. 任务如果持续产生新步骤，缺少步数、时间和 token budget 会导致失控循环。
5. 外部工具即使返回成功，也可能只代表请求被接收；业务完成状态还需二次验证。

这些边界解释了为什么生产 Agent 需要 Harness、Evals 和可观测性，而不只是一段 Prompt。

## 9. 上线检查项

1. **工具契约**：参数 Schema、返回结构、超时、错误分类和幂等策略明确。
2. **数据权限**：RAG 在检索阶段执行 metadata 和用户权限过滤。
3. **状态恢复**：对话、任务、工具结果、checkpoint 和 artifact 分层存储。
4. **执行预算**：限制步数、时间、token、费用和并发。
5. **可观测性**：模型调用、工具调用、trace、latency、cost 和错误可关联。
6. **评测**：固定 eval dataset，覆盖回答质量、检索召回、工具参数和任务完成率。
7. **失败处理**：重试、退避、降级、终止、恢复和转人工有明确条件。
8. **审批与权限**：删除、发送、付款、发布等副作用动作执行前检查权限并人审。
9. **安全**：处理 prompt injection、越权访问、敏感数据泄露和供应链风险。
10. **审计**：关键输入、决策、批准、工具结果和业务状态变更可追溯。

## 10. 常见误区

1. 把模型品牌、框架、协议、平台和完整产品放在同一层比较。
2. 只优化 Prompt，不设计状态、工具契约、失败处理和评测。
3. 把 RAG 当成 Agent，忽略 Loop 与工具副作用。
4. 把 MCP 当成 Agent 框架，或把 CLI 当成完整 Agent。
5. 只看最终回答，不检查检索证据、工具参数和执行轨迹。
6. 单 Agent 尚未稳定就增加多 Agent，造成通信成本和错误放大。
7. 依赖框架默认行为，没有定义停止条件、预算和审批边界。
8. 用 Demo 成功代替回归测试和线上可观测性。

## 11. 本章要点

1. 通用大模型把此前分散在规则推理、搜索、感知、预测和语言处理系统中的能力汇入可跨任务复用的预训练模型。
2. 主链是：目标 → 上下文装配 → Tokenization / LLM → Agent Loop → 能力接口 → 外部工具 → 验证与反馈。
3. Harness 把模型、Loop、知识、工具、状态、安全和评测装配成可运行系统。
4. RAG、Skills、Spec、Wiki 和 Memory 都服务于上下文，但职责不同。
5. Agent 框架需要按控制权、状态位置、语言、恢复语义和部署边界分类；第 11 章选取三个框架做实验，生态索引覆盖更多框架。
6. Claude Code、Codex、OpenCode、OpenHands 属于完整 Coding Agent 产品；MCP、A2A、Agent Skills 属于协议或规范。
7. Hugging Face、百炼、方舟、AutoDL、可灵和 GigaTok 分别落在模型生态、平台、算力或多模态层。
8. 生产质量来自工具契约、状态恢复、权限、人审、评测、追踪和审计。

## 参考文献

[1] SILENTVIEW. GigaTok: Scaling visual tokenizers to 3 billion parameters[EB/OL]. GitHub, 2025[2026-07-27]. https://github.com/SilentView/GigaTok.

[2] LEWIS P, PEREZ E, PIKTUS A, et al. Retrieval-augmented generation for knowledge-intensive NLP tasks[EB/OL]. arXiv, 2020[2026-07-27]. https://arxiv.org/abs/2005.11401.

[3] AGENT SKILLS. Specification[EB/OL]. [2026-07-27]. https://agentskills.io/specification.

[4] MODEL CONTEXT PROTOCOL. Architecture overview[EB/OL]. [2026-07-27]. https://modelcontextprotocol.io/docs/learn/architecture.

[5] HUGGING FACE. Hugging Face Hub documentation[EB/OL]. [2026-07-27]. https://huggingface.co/docs/hub/en/index.

[6] 阿里云. 什么是阿里云百炼[EB/OL]. [2026-07-27]. https://help.aliyun.com/zh/model-studio/what-is-model-studio.

[7] 火山引擎. 方舟大模型服务平台文档[EB/OL]. [2026-07-27]. https://www.volcengine.com/docs/82379/.

[8] 智谱 AI. GLM 开放平台介绍[EB/OL]. [2026-07-27]. https://docs.bigmodel.cn/cn/guide/start/introduction.

[9] AUTODL. AutoDL 帮助文档[EB/OL]. [2026-07-27]. https://www.autodl.com/docs/.

[10] 快手科技. 可灵 AI 相关业务进展[EB/OL]. [2026-07-27]. https://ir.kuaishou.com/node/11111/pdf.

[11] LANGCHAIN. LangGraph overview[EB/OL]. [2026-07-27]. https://docs.langchain.com/oss/python/langgraph/overview.

[12] CREWAI. CrewAI documentation[EB/OL]. [2026-07-27]. https://docs.crewai.com/.

[13] MICROSOFT. AutoGen[EB/OL]. GitHub[2026-07-27]. https://github.com/microsoft/autogen.

[14] MICROSOFT. Migrating from AutoGen to Microsoft Agent Framework[EB/OL]. [2026-07-27]. https://learn.microsoft.com/en-us/agent-framework/migration-guide/from-autogen/.

[15] ANTHROPIC. How Claude Code works[EB/OL]. [2026-07-27]. https://code.claude.com/docs/en/how-claude-code-works.

[16] OPENAI. Codex[EB/OL]. [2026-07-27]. https://openai.com/codex/get-started/.

[17] OPENCODE. OpenCode documentation[EB/OL]. [2026-07-27]. https://opencode.ai/docs/.

[18] OPENAI. OpenAI Agents SDK: Agents[EB/OL]. [2026-07-27]. https://openai.github.io/openai-agents-python/agents/.

[19] ANTHROPIC. Claude Agent SDK overview[EB/OL]. [2026-07-27]. https://code.claude.com/docs/en/agent-sdk/overview.

[20] GOOGLE. Agent Development Kit[EB/OL]. [2026-07-27]. https://adk.dev/.

[21] AWS. Strands Agents documentation[EB/OL]. [2026-07-27]. https://strandsagents.com/docs/user-guide/quickstart/overview/.

[22] PYDANTIC. Pydantic AI agents[EB/OL]. [2026-07-27]. https://pydantic.dev/docs/ai/core-concepts/agent/.

[23] HUGGING FACE. smolagents: Agents[EB/OL]. [2026-07-27]. https://huggingface.co/docs/smolagents/main/reference/agents.

[24] LLAMAINDEX. Agents[EB/OL]. [2026-07-27]. https://developers.llamaindex.ai/python/framework/module_guides/deploying/agents/.

[25] DEEPSET. Haystack Agent[EB/OL]. [2026-07-27]. https://docs.haystack.deepset.ai/docs/agent.

[26] AGNO. What are Agents?[EB/OL]. [2026-07-27]. https://docs.agno.com/agents/overview.

[27] BEEAI. BeeAI Framework introduction[EB/OL]. [2026-07-27]. https://framework.beeai.dev/introduction/welcome.

[28] CAMEL-AI. CAMEL introduction[EB/OL]. [2026-07-27]. https://docs.camel-ai.org/get_started/introduction.

[29] MASTRA. TypeScript AI framework for agents and apps[EB/OL]. [2026-07-27]. https://mastra.ai/.

[30] VERCEL. AI SDK[EB/OL]. [2026-07-27]. https://vercel.com/ai-sdk.

[31] LETTA. Letta documentation[EB/OL]. [2026-07-27]. https://docs.letta.com/.

[32] AG2. AG2 multi-agent framework[EB/OL]. [2026-07-27]. https://www.ag2.ai/.

[33] LANGROID. Langroid multi-agent programming framework[EB/OL]. [2026-07-27]. https://langroid.github.io/langroid/.

[34] FOUNDATIONAGENTS. MetaGPT[EB/OL]. GitHub[2026-07-27]. https://github.com/FoundationAgents/MetaGPT.

[35] MICROSOFT. Microsoft Agent Framework overview[EB/OL]. [2026-07-27]. https://learn.microsoft.com/en-us/agent-framework/overview/.

[36] TURING A M. Computing machinery and intelligence[J]. Mind, 1950, 59(236): 433-460. DOI:10.1093/mind/LIX.236.433.

[37] MCCARTHY J, MINSKY M L, ROCHESTER N, et al. A proposal for the Dartmouth summer research project on artificial intelligence[EB/OL]. 1955[2026-08-07]. https://www-formal.stanford.edu/jmc/history/dartmouth/dartmouth.html.

[38] SHORTLIFFE E H, DAVIS R, AXLINE S G, et al. Computer-based consultations in clinical therapeutics: explanation and rule acquisition capabilities of the MYCIN system[J]. Computers and Biomedical Research, 1975, 8(4): 303-320. DOI:10.1016/0010-4809(75)90009-9.

[39] LECUN Y, BOTTOU L, BENGIO Y, et al. Gradient-based learning applied to document recognition[J]. Proceedings of the IEEE, 1998, 86(11): 2278-2324. DOI:10.1109/5.726791.

[40] CAMPBELL M, HOANE A J, HSU F H. Deep Blue[J]. Artificial Intelligence, 2002, 134(1-2): 57-83. DOI:10.1016/S0004-3702(01)00129-1.

[41] KRIZHEVSKY A, SUTSKEVER I, HINTON G E. ImageNet classification with deep convolutional neural networks[C]//Advances in Neural Information Processing Systems 25. 2012: 1097-1105.

[42] SILVER D, HUANG A, MADDISON C J, et al. Mastering the game of Go with deep neural networks and tree search[J]. Nature, 2016, 529: 484-489. DOI:10.1038/nature16961.

[43] VASWANI A, SHAZEER N, PARMAR N, et al. Attention is all you need[C]//Advances in Neural Information Processing Systems 30. 2017.

[44] DEVLIN J, CHANG M W, LEE K, et al. BERT: pre-training of deep bidirectional transformers for language understanding[C]//Proceedings of NAACL-HLT 2019. Minneapolis: Association for Computational Linguistics, 2019: 4171-4186. DOI:10.18653/v1/N19-1423.

[45] BROWN T B, MANN B, RYDER N, et al. Language models are few-shot learners[C]//Advances in Neural Information Processing Systems 33. 2020: 1877-1901.

[46] OUYANG L, WU J, JIANG X, et al. Training language models to follow instructions with human feedback[EB/OL]. arXiv, 2022[2026-08-07]. https://arxiv.org/abs/2203.02155.

[47] OPENAI. Introducing ChatGPT[EB/OL]. 2022-11-30[2026-08-07]. https://openai.com/index/chatgpt/.

[48] NEWELL A, SHAW J C, SIMON H A. Empirical explorations of the Logic Theory Machine: a case study in heuristic[C]//Proceedings of the Western Joint Computer Conference. Los Angeles: ACM, 1957: 218-230. DOI:10.1145/1455567.1455605.